
**How to use the widget:**

1. Replace `"path_to_your_model"` with the actual path to your fine-tuned Whisper model.
2. Run the code in a Jupyter Notebook.
3. Press and hold the spacebar to record; a red waveform will indicate recording.
4. Release the spacebar to stop recording and transcribe the audio using your fine-tuned model.
5. The transcription appears in the output area below the visualization.

**Notes:**
- The visualization uses a simulated waveform (red during recording, blue when idle) with a dark background for clarity on both light and dark themes.
- The widget uses your model’s pipeline configuration, matching your example code.
- Ensure `pyaudio`, `librosa`, `transformers`, `torch`, and `ipywidgets` are installed.
- The temporary audio file (`temp_recording.wav`) is saved in the working directory and overwritten with each recording.
- If you encounter issues with the model loading, verify the model path and dependencies.



In [7]:
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML
import pyaudio
import wave
import numpy as np
import matplotlib.pyplot as plt
from threading import Thread
import time
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
import librosa
import os



ModuleNotFoundError: No module named 'librosa'

In [ ]:

# Audio recording settings
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
WAVE_OUTPUT_FILENAME = "temp_recording.wav"
MODEL_PATH = "model/"  # Update with your model path
# Global variables for recording
recording = False
frames = []

# Widget setup
output = widgets.Output()
status_label = widgets.Label(value="Press and hold spacebar to record")
visualization = widgets.Output()

# Load your fine-tuned Whisper model and processor
model_path = MODEL_PATH  # Replace with your model path
print("🔹 Loading fine-tuned model...")
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_path)
    processor = WhisperProcessor.from_pretrained(model_path)
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=0 if torch.cuda.is_available() else -1,
        generate_kwargs={"forced_decoder_ids": None}
    )
    print("✅ Fine-tuned model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# Function to record audio
def record_audio():
    global recording, frames
    audio = pyaudio.PyAudio()
    stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
    frames = []
    
    while recording:
        data = stream.read(CHUNK, exception_on_overflow=False)
        frames.append(data)
    
    stream.stop_stream()
    stream.close()
    audio.terminate()
    
    # Save the recorded audio
    wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(audio.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))
    wf.close()

# Function to transcribe audio
def transcribe_audio():
    with output:
        output.clear_output()
        print("Transcribing...")
        try:
            # Load and preprocess audio
            audio, sr = librosa.load(WAVE_OUTPUT_FILENAME, sr=16000)
            # Generate prediction
            result = pipe(audio)
            transcription = result['text'].strip()
            print("Transcription:", transcription)
        except Exception as e:
            print(f"❌ Error transcribing audio: {e}")

# Function to update visualization
def update_visualization():
    with visualization:
        visualization.clear_output(wait=True)
        plt.figure(figsize=(4, 1))
        if recording:
            # Simulate waveform for visualization
            t = np.linspace(0, 0.5, CHUNK)
            y = np.sin(2 * np.pi * 440 * t) * np.random.rand(CHUNK)  # Simulated audio signal
            plt.plot(t, y, color='#FF5555')  # Vibrant red for recording
            plt.title("Recording...", color='#FFFFFF')
        else:
            plt.plot([0], [0], color='#5555FF')  # Blue for idle
            plt.title("Not Recording", color='#FFFFFF')
        plt.axis('off')
        plt.gca().set_facecolor('#222222')  # Dark background
        plt.gcf().set_facecolor('#222222')
        plt.show()

# Thread to continuously update visualization
def visualization_thread():
    while True:
        update_visualization()
        time.sleep(0.1)

# Start visualization thread
Thread(target=visualization_thread, daemon=True).start()

# JavaScript to detect spacebar press and release
js_code = """
document.addEventListener('keydown', function(event) {
    if (event.code === 'Space') {
        IPython.notebook.kernel.execute('recording = True');
        IPython.notebook.kernel.execute('Thread(target=record_audio).start()');
        IPython.notebook.kernel.execute('status_label.value = "Recording..."');
    }
});
document.addEventListener('keyup', function(event) {
    if (event.code === 'Space') {
        IPython.notebook.kernel.execute('recording = False');
        IPython.notebook.kernel.execute('status_label.value = "Processing..."');
        IPython.notebook.kernel.execute('transcribe_audio()');
        IPython.notebook.kernel.execute('status_label.value = "Press and hold spacebar to record"');
    }
});
"""

# Display JavaScript
display(Javascript(js_code))

# Display widgets
display(status_label)
display(visualization)
display(output)